# Urban Similarity Model Analysis — Chicago (`chicago_model_v1`)

This notebook builds the Chicago-only urban similarity model over the 77 Community Areas from the frozen contract `analysis/config/chicago_model_v1.json`. It reads the 12 accepted feature families, runs the two pre-registered checks of the final review, transforms and scales the features, computes the dissimilarity between every pair of areas, and reports three runs:

1. **R1 — equal weights**: every family has the same budget.
2. **R2 — PCA weights**: distance in the principal components of R1 that explain 90% of its variance.
3. **R3 — without correlated features**: scalar columns are dropped until no pair correlates at |ρ| ≥ 0.70.

It mirrors `analysis/model_analysis.ipynb` (São Paulo): top-10 table, contribution bars, map, radar, PCA variance, loadings, PCA scatter, truncated-PCA rank shifts, PCA-weighted ranking and Ward clustering. Target-based views use a **display reference** (`TARGET`, default the Loop, `CHI:32`); the model itself is target-free (decision J-1), and every area's nearest neighbours are saved. All rules were fixed before any distance was computed ([MODEL_PLAN](../docs/chicago/MODEL_PLAN.md), final review). The calculations live in `analysis/scripts/chicago_model.py`, so the notebook and the report use identical code.

## Math and techniques

**Transforms (J-5).** Natural log for strictly positive ratio-scale quantities (job, resident, street and parcel densities, transit access, building height, block elongation). Identity for shares, indices and log-ratios (footprint coverage, major-street share, block compactness, block size already in ln m², U6 log intensity and log-ratios, U1 land shares).

**Scaling (J-6).** Each transformed scalar column is standardized on the 77 areas, $z_{dj} = (t(x_{dj}) - \bar t_j)/s_j$ with the population standard deviation, and capped at ±3. The standard deviation keeps the tail that carries the real variation of grid-uniform columns; the cap stops one extreme area from dominating.

**Composition (U6).** The five activity classes are compared with the **Aitchison distance**: Euclidean distance on centred log-ratios $\mathrm{clr}_k = \ln(n_k + 0.5) - \frac{1}{5}\sum_j \ln(n_j + 0.5)$, left unscaled because scaling log-ratios one by one would break compositional geometry.

**Blocks and calibration (J-7).** A family is one block (U6 is two: intensity and composition, each with half of U6's budget). For a block $b$ with $c_b$ columns, $m_b(d,e) = \frac{1}{c_b}\sum_j (x_{dj} - x_{ej})^2$, divided by its median positive pairwise value $\beta_b$ so that every block contributes on the same scale.

**Distance.** With family budget $w_f$ (equal, $1/12$) and block share $s_b$:
$$D(d,e) = \sqrt{\frac{\sum_b w_{f(b)}\, s_b\, m_b(d,e)/\beta_b}{\sum_{b \text{ available for } d,e} w_{f(b)}\, s_b}}$$
The denominator is 1 except for pairs involving O'Hare, whose U1 is missing (J-8: the pair uses the families both areas have).

**PCA (R2).** Principal components of the R1 geometry are computed from $D$ (principal-coordinate analysis). When every area has every family this is exactly PCA of the centred weighted embedding, as in `model_analysis`. The fewest components reaching 90% of the variance are kept and distance is Euclidean on their unwhitened scores.

**Correlation pruning (R3).** Spearman among the scalar columns; while any pair has |ρ| ≥ 0.70, drop the column with the most such pairs (ties: higher mean |ρ|, then later contract position).

## Attribute data dictionary

The model uses **20 columns in 12 families** across three domains. Every value was accepted family by family with pre-registered checks (see each family's results README).

**Street morphology (M)**
* **M1 street density** `m1_km_per_km2`: Overture ten-class street length per km² of land.
* **M3 block size** `ov_m3_wmedian_ln_m2`: area-weighted median of ln(face area) of the street faces; the size of the block in which a typical m² of land sits.
* **M4 block shape** `ov_m4_wmedian_compactness`, `ov_m4_wmedian_elongation`: area-weighted medians of 4πA/P² and of longest ÷ shortest side.
* **M6 street hierarchy** `m6_major_share`: share of street length in motorway, trunk, primary and secondary.
* **M7 mapped ground-parcel density** `m7_parcels_per_km2`: Cook 2024 ground parcels per km² of land.

**Built form (B)**
* **B1 footprint coverage** `B1_coverage_land`: union of Overture building footprints ÷ land.
* **BV vertical form** `bv_height_built_m`: GHSL built-surface-weighted building height (m).

**Use and activity (U)**
* **U2 job density** `u2_jobs_land_km2`: LODES 2023 jobs per km² of land.
* **U3 resident density** `u3_acs_land_km2`: ACS 2020–2024 residents per km² of land.
* **U4 public transport accessibility** `u4_ptai_avg_resident`: PTAL Access Index (bus, 'L', Metra) averaged over residents.
* **U6 activity composition** `u6_log_intensity` + five `u6_clr_city_*`: establishments per 100 dwellings (log) and the mix of food & drink, retail, services & offices, making & storing, institutions (Overture places).
* **U1 land-use composition** `p_residential`, `p_industrial`, `p_institutional`: shares of classified occupied land (CMAP 2023); admitted at the final review (J-3); O'Hare missing (4.9% classified land).

In [ ]:
import sys, json
from pathlib import Path
from math import pi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

base_dir = Path.cwd() if (Path.cwd() / 'config/chicago_model_v1.json').exists() else Path.cwd() / 'analysis'
sys.path.insert(0, str(base_dir / 'scripts'))
import chicago_model as cm

TARGET = 'CHI:32'                      # display reference only (J-1: the model is target-free)
OUT = cm.OUT; FIG = OUT / 'figures'; TAB = OUT / 'tables'
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.savefig(FIG / f'{name}.png', dpi=150, bbox_inches='tight')

In [ ]:
# Frozen inputs: each family table is verified against the SHA-256 recorded at its acceptance.
contract, df, subs, names, families = cm.load_inputs()
ids = df.index.tolist(); n = len(ids)
target_name = names[TARGET]
print(f"Contract {contract['contract_version']}: {contract['status']}; fit_authorized={contract['fit_authorized']}")
print(f"Loaded {n} Community Areas, {sum(len(c) for c in families.values())} columns in {len(families)} families.")
print("Missing values:", df.isna().sum()[df.isna().sum() > 0].to_dict(), "->", df.index[df.isna().any(axis=1)].tolist())
display(df.assign(name=names).set_index('name', append=True).head())

## 1. Pre-registered checks before the fit

**J-3, U1 land-share admission.** A U1 share enters only if its largest absolute Spearman correlation with the U6 coordinates, B1 and M1 is below 0.70 in **both** cities (São Paulo values from the same tables). **J-4, redundancy.** Any two families whose scalar columns correlate at |ρ| ≥ 0.90 in Chicago would share one budget.

In [ ]:
adm = cm.u1_admission()
display(adm)
scalar_cols = [c for cs in families.values() for c in cs if c not in cm.CLR]
rho, groups = cm.redundancy(df, families)
print("J-4 merged groups (|rho| >= 0.90):", groups or "none")
adm.to_csv(TAB / 'j3_u1_admission.csv'); rho.to_csv(TAB / 'spearman_scalar_columns.csv')

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(rho, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, ax=ax, annot_kws={'size': 7})
ax.set_title('Spearman correlation among scalar model columns (77 areas)')
save(fig, 'spearman_heatmap'); plt.show()

## 2. Transform and scale (J-5, J-6)

The table shows each column's transform, mean, standard deviation, median and IQR on the transformed scale, and the areas whose standardized value is capped at ±3. Note the near-zero IQR of the block-shape columns: Chicago's uniform grid is why the IQR scaling of the São Paulo model was not used.

In [ ]:
sc = cm.scaling_table(df, scalar_cols)
sc['capped'] = sc.capped.map(lambda s: ', '.join(names[x] for x in s.split(', ')) if s else '')
sc.round(4).to_csv(TAB / 'scaling.csv')
display(sc.round(3))

## 3. Run R1 — equal weights

Twelve families, each with budget 1/12; U6 splits its budget between intensity and composition. Each block is divided by its median positive pairwise value (calibration).

In [ ]:
bl = cm.blocks(df, families)
bud = cm.equal_budgets(families, groups)
D1, contrib1 = cm.distance(bl, bud)
pd.DataFrame(D1, index=ids, columns=ids).to_csv(TAB / 'distance_R1.csv')
display(pd.DataFrame([dict(family=b['family'], domain=cm.DOMAIN[b['family']], block=b['block'], columns=len(b['columns']),
                           budget=bud[b['family']] * b['share'], calibration=b['cal']) for b in bl]).round(4))

res = cm.rank_target(D1, ids, TARGET); res['name'] = names
print(f"Top 10 most similar Community Areas to {target_name} (R1, equal weights):")
display(res[['name', 'distance', 'rank']].head(10))
nb1 = cm.neighbours(D1, ids); nb1['name'] = nb1.unit_id.map(names); nb1['neighbour_name'] = nb1.neighbour.map(names)
nb1.to_csv(TAB / 'neighbours_R1.csv', index=False)

### Which families drive the distances (all pairs)
Equal budgets make every family count the same on a *typical* pair, because each block is divided by its median pairwise value. The table shows what each family actually contributes to $D^2$ across all 2,926 pairs. A family whose values are nearly identical in most areas but extreme in a few (a narrow spike with a tail) has a small median and therefore weighs heavily whenever a tail area is involved.

In [ ]:
shares = cm.contribution_shares(contrib1, D1)
shares.round(3).to_csv(TAB / 'family_contribution_shares_R1.csv')
display(shares.round(3))
fig, ax = plt.subplots(figsize=(9, 4))
shares.mean_share.plot(kind='bar', color='steelblue', ax=ax)
ax.axhline(1 / len(families), color='r', linestyle='--', label='nominal equal share (1/12)')
ax.set_ylabel('Mean share of D² over all pairs'); ax.set_title('R1: actual contribution of each family'); ax.legend()
save(fig, 'family_contribution_shares'); plt.show()

## Visualizing results
### Breakdown of squared distance for the top 10 neighbours
Each bar is the squared distance $D^2$ between the reference and a neighbour, split into the families' contributions.

In [ ]:
top_10 = res.head(10).index
ti = ids.index(TARGET)
plot = pd.DataFrame({f: [contrib1[f][ti, ids.index(d)] for d in top_10] for f in families},
                    index=[f"{names[d]} ({d})" for d in top_10])
fig, ax = plt.subplots(figsize=(12, 6))
plot.plot(kind='bar', stacked=True, colormap='tab20', ax=ax)
ax.set_title(f'Squared distance contributions for the top 10 neighbours of {target_name} (R1)')
ax.set_ylabel('Squared distance (total is D²)'); ax.set_xlabel('Community Area')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left'); plt.xticks(rotation=45, ha='right')
save(fig, 'contributions_top10_R1'); plt.show()

### Worked example: one distance, family by family
The reference and its closest match: each family's part of $D^2$ (budget × calibrated block value) and its share. $D$ is the square root of the total.

In [ ]:
j = ids.index(top_10[0])
we = pd.DataFrame({'contribution_to_D2': {f: contrib1[f][ti, j] for f in families}})
we['share'] = we.contribution_to_D2 / we.contribution_to_D2.sum()
we.loc['total'] = [we.contribution_to_D2.sum(), 1.0]
print(f"{target_name} vs {names[top_10[0]]}: D = sqrt({we.loc['total', 'contribution_to_D2']:.4f}) = {D1[ti, j]:.4f}")
we.round(4).to_csv(TAB / 'worked_example_reference_vs_top_match.csv')
display(we.round(4))

In [ ]:
gdf = gpd.read_file(base_dir / 'data/Chicago/Boundaries_-_Community_Areas_20260831.geojson')
gdf['unit_id'] = 'CHI:' + gdf.area_numbe.astype(int).astype(str).str.zfill(2)
gdf['name'] = gdf.unit_id.map(names)
gdf = gdf.merge(res[['distance', 'rank']], left_on='unit_id', right_index=True, how='left')

m = gdf.explore(tiles='OpenStreetMap', tooltip=['name', 'distance', 'rank'],
                style_kwds={'weight': 1, 'color': 'gray', 'fillOpacity': 0.1, 'fillColor': '#e0e0e0'})
m = gdf[gdf['rank'] <= 10].explore(m=m, style_kwds={'weight': 3, 'fillOpacity': 0.4, 'color': 'red', 'fillColor': 'red'},
                                   tooltip=['name', 'rank', 'distance'])
m = gdf[gdf.unit_id == TARGET].explore(m=m, style_kwds={'weight': 4, 'fillOpacity': 0.4, 'color': 'blue', 'fillColor': 'blue'}, tooltip=['name'])
m

In [ ]:
# Static version of the map (saved for the report): distance to the reference, top 10 outlined.
fig, ax = plt.subplots(figsize=(7, 9))
g = gdf.to_crs(26916)
g.plot(column='distance', cmap='viridis_r', legend=True, ax=ax, missing_kwds={'color': 'navy'},
       legend_kwds={'label': f'R1 distance to {target_name}', 'shrink': .6})
g[g['rank'] <= 10].boundary.plot(ax=ax, color='red', linewidth=1.8)
g[g.unit_id == TARGET].plot(ax=ax, color='navy')
ax.set_axis_off(); ax.set_title(f'R1 distance to {target_name} (navy); top 10 outlined in red')
save(fig, 'map_distance_R1'); plt.show()

### Radar chart comparison: reference vs top match
Selected raw-variable mean/SD z-scores (one per family; U6 shown by its intensity), not the fitted distance coordinates; use the contribution bars for the full metric.

In [ ]:
top_match_id = top_10[0]; top_match_name = names[top_match_id]
radar_cols = {'M1: Street dens': 'm1_km_per_km2', 'M3: Block size': 'ov_m3_wmedian_ln_m2', 'M4: Compactness': 'ov_m4_wmedian_compactness',
              'M6: Major share': 'm6_major_share', 'M7: Parcel dens': 'm7_parcels_per_km2', 'B1: Bldg cover': 'B1_coverage_land',
              'BV: Height': 'bv_height_built_m', 'U2: Job dens': 'u2_jobs_land_km2', 'U3: Pop dens': 'u3_acs_land_km2',
              'U4: Transit': 'u4_ptai_avg_resident', 'U6: Activity': 'u6_log_intensity', 'U1: Residential land': 'p_residential'}
cats = list(radar_cols); N = len(cats)
raw = df[list(radar_cols.values())].astype(float)
z = (raw - raw.mean()) / raw.std()
v1 = z.loc[TARGET].tolist(); v2 = z.loc[top_match_id].tolist()
v1 += v1[:1]; v2 += v2[:1]
ang = [k / float(N) * 2 * pi for k in range(N)]; ang += ang[:1]
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(pi / 2); ax.set_theta_direction(-1)
plt.xticks(ang[:-1], cats, size=10); ax.set_rlabel_position(0)
lo, hi = np.floor(min(v1 + v2)) - .5, np.ceil(max(v1 + v2)) + .5
ax.set_ylim(lo, hi)
ax.plot(ang, v1, linewidth=2, label=f'{target_name} (reference)'); ax.fill(ang, v1, alpha=.1)
ax.plot(ang, v2, linewidth=2, label=f'{top_match_name} (top match)'); ax.fill(ang, v2, alpha=.1)
ax.set_title(f'Selected raw-variable z-score profile: {target_name} vs {top_match_name}', size=14, y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
save(fig, 'radar_R1'); plt.show()

## 4. Principal component analysis (PCA) and attribute variance

The model is **unsupervised** (there is no outcome to predict), so feature importance comes from the structure of the data. PCA of the R1 geometry shows which directions carry the variation across Chicago's 77 areas. Loadings are reported as correlations between each component and the model columns.

In [ ]:
coords = cm.coordinates(bl, ids)
P = cm.pca(D1, ids, coords)
pca_scores = P['scores']; loadings = P['loadings']
print(f"Negative-eigenvalue mass (non-Euclidean part from O'Hare's available-case distances): {P['negative_mass']:.2e}")
fig, ax = plt.subplots(figsize=(8, 4))
k_all = len(P['cumulative'])
ax.plot(range(1, k_all + 1), P['cumulative'], marker='o', linestyle='--')
ax.axhline(0.90, color='r'); ax.text(1, .92, '90% variance threshold', color='red')
ax.set_title('PCA cumulative explained variance (R1 geometry)'); ax.set_xlabel('Number of components'); ax.set_ylabel('Cumulative explained variance')
ax.set_xticks(range(1, k_all + 1))
save(fig, 'pca_cumulative_variance'); plt.show()
pd.DataFrame({'explained': P['explained'], 'cumulative': P['cumulative']}, index=pca_scores.columns).to_csv(TAB / 'pca_variance.csv')
loadings.to_csv(TAB / 'pca_loadings.csv')

In [ ]:
print("PCA loadings (correlation with the component) — PC1:")
display(loadings.sort_values(by='PC1', key=abs, ascending=False)['PC1'].to_frame().head(10).round(3))

### PCA 2D visualization
All areas projected onto the first two principal components; the reference and its R1 top 10 are highlighted.

In [ ]:
pca_df = pca_scores[['PC1', 'PC2']].copy(); pca_df['name'] = names
fig, ax = plt.subplots(figsize=(10, 8))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', color='lightgrey', s=50, label='Other areas', ax=ax)
sns.scatterplot(data=pca_df.loc[top_10], x='PC1', y='PC2', color='orange', s=100, edgecolor='black', label='Top 10 matches (R1)', ax=ax)
sns.scatterplot(data=pca_df.loc[[TARGET]], x='PC1', y='PC2', color='red', s=250, marker='*', edgecolor='black', label=target_name, ax=ax)
for u in [TARGET, top_match_id, 'CHI:76']:
    ax.text(pca_df.loc[u, 'PC1'] + .05, pca_df.loc[u, 'PC2'] + .05, names[u], weight='bold')
ax.set_title('Community Areas projected onto PC1 and PC2')
save(fig, 'pca_scatter'); plt.show()

In [ ]:
def color_shift(val):
    if pd.isna(val):
        return ''
    return f"color: {'green' if val > 0 else 'red' if val < 0 else 'black'}"

baseline_ranks = res['rank']
def truncated(k):
    sc_k = pca_scores.iloc[:, :k].to_numpy()
    d = np.linalg.norm(sc_k - sc_k[ti], axis=1)
    r = pd.DataFrame({'pca_distance': d, 'name': names.values}, index=ids).drop(TARGET).sort_values('pca_distance', kind='stable')
    r['pca_rank'] = r.pca_distance.rank(method='min').astype(int)
    r['rank_shift'] = baseline_ranks - r.pca_rank
    t = r[['name', 'pca_distance', 'pca_rank', 'rank_shift']].head(10).copy(); t['pca_distance'] = t.pca_distance.round(4)
    return t

### Truncated PCA distance: k=1 dimension

In [ ]:
display(truncated(1).style.map(color_shift, subset=['rank_shift']))

### Truncated PCA distance: k=2 dimensions

In [ ]:
display(truncated(2).style.map(color_shift, subset=['rank_shift']))

### Truncated PCA distance: k=3 dimensions

In [ ]:
display(truncated(3).style.map(color_shift, subset=['rank_shift']))

### Truncated PCA distance: k=4 dimensions

In [ ]:
display(truncated(4).style.map(color_shift, subset=['rank_shift']))

### Truncated PCA distance: k=5 dimensions

In [ ]:
display(truncated(5).style.map(color_shift, subset=['rank_shift']))

### Truncated PCA distance: k=6 dimensions

In [ ]:
display(truncated(6).style.map(color_shift, subset=['rank_shift']))

## 5. Run R2 — PCA weights

Retaining the components that explain 90% of the variance and measuring Euclidean distance along them weights the similarity by the dominant structural directions of Chicago. The PCA is fitted to the already equal-family-weighted geometry, so it re-weights directions, not the original assumptions.

### Interpreting the PCA axes

In [ ]:
for pc, title in (('PC1', 'Dominant structural axis'), ('PC2', 'Secondary structural axis')):
    l = loadings[pc].sort_values(ascending=False)
    print(f"--- INTERPRETING {pc} ({title}) ---")
    print("Strongest positive drivers:"); print(l.head(3).round(3).to_string())
    print("Strongest negative drivers:"); print(l.tail(3).sort_values().round(3).to_string()); print()
k = P['k']; D2 = P['D']
print(f"Retaining {k} principal components to capture >= 90% of the variance ({P['cumulative'][k - 1]:.1%}).")
pd.DataFrame(D2, index=ids, columns=ids).to_csv(TAB / 'distance_R2.csv')
nb2 = cm.neighbours(D2, ids); nb2['name'] = nb2.unit_id.map(names); nb2['neighbour_name'] = nb2.neighbour.map(names)
nb2.to_csv(TAB / 'neighbours_R2.csv', index=False)
res2 = cm.rank_target(D2, ids, TARGET); res2['name'] = names
print(f"\n--- TOP 10 MATCHES TO {target_name.upper()} USING PCA-WEIGHTED DISTANCE (R2) ---")
display(res2[['name', 'distance', 'rank']].head(10))
comparison = res2[['name', 'rank']].head(10).rename(columns={'rank': 'pca_rank'})
comparison['equal_weight_rank'] = res.loc[comparison.index, 'rank']
comparison['rank_shift'] = comparison.equal_weight_rank - comparison.pca_rank
print("Comparison of ranks (equal weights vs PCA):")
display(comparison.style.map(color_shift, subset=['rank_shift']))

## 6. Run R3 — without correlated features

Scalar columns are dropped one at a time while any pair correlates at |ρ| ≥ 0.70 (rule fixed before the run). The U6 composition stays: it is a composition, not a scalar column. Families that lose all their columns leave the model; the rest keep equal budgets.

In [ ]:
fam3, prune_log = cm.prune(df, families)
display(prune_log)
prune_log.to_csv(TAB / 'r3_pruning_log.csv', index=False)
print("R3 families:", {f: c for f, c in fam3.items()})
kept = [c for cs in fam3.values() for c in cs if c not in cm.CLR]
r3 = df[kept].corr(method='spearman')
print(f"Largest remaining |rho| among kept scalar columns: {r3.where(~np.eye(len(r3), dtype=bool)).abs().max().max():.3f}")
bl3 = cm.blocks(df, fam3); bud3 = cm.equal_budgets(fam3)
D3, contrib3 = cm.distance(bl3, bud3)
pd.DataFrame(D3, index=ids, columns=ids).to_csv(TAB / 'distance_R3.csv')
nb3 = cm.neighbours(D3, ids); nb3['name'] = nb3.unit_id.map(names); nb3['neighbour_name'] = nb3.neighbour.map(names)
nb3.to_csv(TAB / 'neighbours_R3.csv', index=False)
res3 = cm.rank_target(D3, ids, TARGET); res3['name'] = names
print(f"\n--- TOP 10 MATCHES TO {target_name.upper()} WITHOUT CORRELATED FEATURES (R3) ---")
display(res3[['name', 'distance', 'rank']].head(10))
comp3 = res3[['name', 'rank']].head(10).rename(columns={'rank': 'r3_rank'})
comp3['equal_weight_rank'] = res.loc[comp3.index, 'rank']
comp3['rank_shift'] = comp3.equal_weight_rank - comp3.r3_rank
display(comp3.style.map(color_shift, subset=['rank_shift']))

## 7. Comparing the three runs (target-free)

Agreement between runs over all 2,926 pairs of areas (Spearman of the distances) and the mean share of each area's top five neighbours that two runs have in common.

In [ ]:
runs = {'R1 equal weights': D1, 'R2 PCA weights': D2, 'R3 without correlated': D3}
rows = []
for a in runs:
    for b in runs:
        if a < b:
            rows.append(dict(run_a=a, run_b=b, **cm.compare(runs[a], runs[b])))
cmp_runs = pd.DataFrame(rows).round(3); cmp_runs.to_csv(TAB / 'run_comparison.csv', index=False)
display(cmp_runs)
side = pd.DataFrame({r: [names[x] for x in cm.rank_target(Dx, ids, TARGET).index[:5]] for r, Dx in runs.items()}, index=range(1, 6))
print(f"Top five neighbours of {target_name} in each run:"); display(side)

## 8. Stability and sensitivities (J-9)

**Weight draws.** 500 draws multiply every family budget by a uniform factor in [0.5, 1.5] (seed 20261006), renormalized. A neighbour is **stable** if it stays in the area's top five in at least 80% of draws.

**Sensitivities.** Each scenario is compared with R1 over all pairs and all areas' top five. All are pre-registered (J-9) except the one marked POST-HOC: calibration by the mean instead of the median pairwise value, added after the family-contribution table showed M3's weight.

In [ ]:
stab = cm.weight_draws(bl, families, bud, ids)
stab['name'] = stab.unit_id.map(names); stab['neighbour_name'] = stab.neighbour.map(names)
stab.to_csv(TAB / 'stability_R1_weight_draws.csv', index=False)
print(f"Stable neighbour relations: {stab.stable.mean():.1%} of the 385 top-five relations; "
      f"areas whose five neighbours are all stable: {(stab.groupby('unit_id').stable.all()).sum()} of 77.")
display(stab[stab.unit_id == TARGET][['rank', 'neighbour_name', 'share_of_draws', 'stable']])
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(stab.share_of_draws, bins=20, color='steelblue'); ax.axvline(cm.STABLE, color='r', linestyle='--', label='80% stability rule')
ax.set_xlabel('Share of weight draws in which a baseline top-5 neighbour stays in the top 5'); ax.set_ylabel('Neighbour relations'); ax.legend()
ax.set_title('Stability of R1 nearest neighbours under 500 weight draws')
save(fig, 'stability_histogram'); plt.show()

In [ ]:
def scenario_D(frame, fams, method='standard', budgets=None):
    b = cm.blocks(frame, fams, method)
    return cm.distance(b, budgets or cm.equal_budgets(fams))[0]

scen = {}
for f in families:                                            # leave-one-family-out
    scen[f'without {f}'] = scenario_D(df, {k: v for k, v in families.items() if k != f})
scen['equal domain budgets'] = cm.distance(bl, cm.domain_budgets(families))[0]
scen['robust median/IQR scaling'] = scenario_D(df, families, 'robust')
scen['rank-normal scaling'] = scenario_D(df, families, 'rank')
scen['Ledoit-Wolf Mahalanobis (no U1)'] = cm.mahalanobis(bl)
scen['R2 PCA weights'] = D2
scen['R3 without correlated'] = D3
for name, (fam, old, new) in cm.SUBSTITUTES.items():
    f2 = {k: [new if c == old else c for c in v] for k, v in families.items()}
    scen[name] = scenario_D(df.join(subs[[new]]), f2)
scen['POST-HOC: mean calibration'] = cm.distance(cm.blocks(df, families, calibration='mean'), bud)[0]   # not pre-registered
u6c = cm.u6_confidence_filtered(ids)
scen['U6 confidence >= 0.3'] = scenario_D(df.drop(columns=u6c.columns).join(u6c), families)

base_top = set(cm.rank_target(D1, ids, TARGET).index[:5])
sens = pd.DataFrame([dict(scenario=k, **cm.compare(v, D1),
                          target_top5_kept=len(base_top & set(cm.rank_target(v, ids, TARGET).index[:5])))
                     for k, v in scen.items()]).round(3)
sens.to_csv(TAB / 'sensitivities.csv', index=False)
display(sens)

## 9. Descriptive check: Ward hierarchical clustering

Ward clustering on the R1 geometry (principal coordinates, all positive dimensions) is a descriptive consistency check, not an independent validation of urban types.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as sch

E = pca_scores.to_numpy()
print("--- WARD CLUSTERING: SILHOUETTE SCORES ---")
sil = {}
for kk in range(3, 9):
    lab = AgglomerativeClustering(n_clusters=kk, linkage='ward').fit_predict(E)
    sil[kk] = silhouette_score(E, lab); print(f"k={kk} clusters: silhouette = {sil[kk]:.4f}")
best_k = max(sil, key=sil.get)
print(f"\nBest tested k by silhouette (descriptive, not independent validation): {best_k}")
clusters = pd.Series(AgglomerativeClustering(n_clusters=best_k, linkage='ward').fit_predict(E), index=ids, name='cluster')
pd.DataFrame({'name': names, 'cluster': clusters, 'R1_rank_to_reference': res['rank']}).to_csv(TAB / 'ward_clusters.csv')
members = pd.DataFrame({'name': names[clusters == clusters[TARGET]]})
members['equal_weight_rank'] = res['rank']; members.loc[TARGET, 'equal_weight_rank'] = 0
print(f"\n{target_name} is in cluster {clusters[TARGET]}. Members:"); display(members.sort_values('equal_weight_rank'))
display(pd.DataFrame({'areas': clusters.value_counts().sort_index(),
                      'members': clusters.groupby(clusters).apply(lambda s: ', '.join(names[s.index][:8]) + (' …' if len(s) > 8 else ''))}))

In [ ]:
fig = plt.figure(figsize=(15, 7))
sch.dendrogram(sch.linkage(E, method='ward'), labels=names.values, leaf_rotation=90, leaf_font_size=8)
plt.title('Ward hierarchical clustering dendrogram (R1 geometry)'); plt.ylabel('Euclidean distance')
save(fig, 'ward_dendrogram'); plt.show()

pca_df['cluster'] = clusters
fig, ax = plt.subplots(figsize=(10, 8))
pal = sns.color_palette('tab10', n_colors=best_k)
for c_ in range(best_k):
    sns.scatterplot(data=pca_df[pca_df.cluster == c_], x='PC1', y='PC2', color=pal[c_], s=60, label=f'Cluster {c_}', ax=ax)
sns.scatterplot(data=pca_df.loc[[TARGET]], x='PC1', y='PC2', color='black', s=250, marker='*', edgecolor='white', label=target_name, ax=ax)
ax.set_title(f'Community Areas on PC1 and PC2, coloured by Ward cluster (k={best_k})')
save(fig, 'pca_clusters'); plt.show()

fig, ax = plt.subplots(figsize=(7, 9))
g = gdf.merge(clusters, left_on='unit_id', right_index=True).to_crs(26916)
g.plot(column='cluster', categorical=True, cmap='tab10', legend=True, ax=ax, edgecolor='white', linewidth=.4)
ax.set_axis_off(); ax.set_title(f'Ward clusters on the map (k={best_k})')
save(fig, 'map_clusters'); plt.show()

## 10. Saved outputs

All tables are in `analysis/results/Chicago/chicago_model_v1_2026_10_06/tables/` and all figures in `figures/`: the three distance matrices and neighbour lists, the pre-registered checks, scaling, PCA variance and loadings, pruning log, run comparison, stability, sensitivities and clusters. The methodology and the reading of these results are in `docs/chicago/MODEL_REPORT.md`.

In [ ]:
print(sorted(p.name for p in TAB.glob('*.csv'))); print(sorted(p.name for p in FIG.glob('*.png')))